In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import rand_score, adjusted_rand_score

In [ ]:
merged = pd.read_csv("../data/merged.csv")
print(merged.columns)

In [ ]:


# ── Features retenues pour le clustering ──────────────────────────────────────
features = [
    'studytime', 'Dalc', 'Walc', 'goout', 'freetime',
    'Medu', 'Fedu', 'famrel', 'health',
    'absences_mat', 'G3_mat', 'G3_por', 'failures_mat', 'failures_por'
]

# Stdardisation des données
df_km = merged[features].dropna().copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_km)

# ── Méthode du coude + silhouette pour choisir k ──────────────────────────────
inertia = []
silhouettes = []
K_range = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertia.append(km.inertia_)
    # silhouettes.append(silhouette_score(X_scaled, labels))

plt.figure(figsize=(6, 4))

plt.plot(K_range, inertia, marker='o', color='#185FA5')

plt.title('Méthode du coude (Inertie)')
plt.xlabel('Nombre de clusters k')
plt.ylabel('Inertie')

plt.grid(alpha=0.3)

plt.show()

# ── Application du K-Means avec k optimal (à ajuster selon les graphes) ───────
k_optimal = 4  # à modifier selon la méthode du coude / silhouette

km_final = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
df_km['cluster'] = km_final.fit_predict(X_scaled)

# ── Visualisation 2D via PCA ───────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df_km['pca1'] = X_pca[:, 0]
df_km['pca2'] = X_pca[:, 1]

plt.figure(figsize=(8, 6))
palette = {0: '#185FA5', 1: '#0F6E56', 2: '#A32D2D', 3: '#BA7517', 4: '#7F77DD'}
for c in sorted(df_km['cluster'].unique()):
    sub = df_km[df_km['cluster'] == c]
    plt.scatter(sub['pca1'], sub['pca2'], label=f'Cluster {c}',
                color=palette[c], alpha=0.6, s=40)
plt.title('Clusters K-Means projetés via PCA (2 composantes)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ── Profil moyen de chaque cluster ────────────────────────────────────────────
profile = df_km.groupby('cluster')[features].mean().round(2)
print("\nProfil moyen par cluster :")
print(profile.to_string())

# ── Heatmap du profil des clusters ────────────────────────────────────────────
profile_norm = (profile - profile.min()) / (profile.max() - profile.min())  # normalisation 0-1
plt.figure(figsize=(12, 4))
sns.heatmap(profile_norm, annot=profile.values, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, cbar_kws={'label': 'Valeur normalisée (0-1)'})
plt.title('Heatmap des profils moyens par cluster (valeurs brutes annotées)')
plt.xlabel('Variables')
plt.ylabel('Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# colonne qui contient vos "vraies" étiquettes de référence.
#variable avec laquelle vous voulez comparer vos clusters 
colonne_cible = 'Medu'

# on récupère les vraies étiquettes en s'assurant de garder les mêmes lignes que celles de df_km (après votre dropna)
y_true = merged.loc[df_km.index, colonne_cible]

# on récupère les prédictions (les clusters générés par votre K-Means)
y_pred = df_km['cluster']

# Calcul de l'indice de Rand (RI)
# L'indice de Rand varie de 0 à 1 (1 = correspondance parfaite)
ri = rand_score(y_true, y_pred)
print(f"Indice de Rand (RI) : {ri:.4f}")

# Calcul de l'indice de Rand Ajusté (ARI)
# L'ARI prend en compte le hasard. Il va de -1 à 1 (1 = parfaite, 0 = aléatoire, <0 = pire que l'aléatoire)
ari = adjusted_rand_score(y_true, y_pred)
print(f"Indice de Rand Ajusté (ARI) : {ari:.4f}")

# Commentaires gemini:

Cette heatmap montre les profils moyens des individus dans chaque cluster. Les couleurs sont normalisées entre 0 et 1 (vert = valeur élevée relativement aux autres clusters pour cette variable, rouge = faible), tandis que les nombres inscrits sont les valeurs réelles moyennes.

Je vais interpréter chaque cluster comme un profil d’étudiants (vu les variables : studytime, absences, G3, failures, etc.).

Variables importantes

Avant l’interprétation :

studytime : temps d’étude
Dalc / Walc : consommation d’alcool semaine / week-end
goout : sorties
freetime : temps libre
Medu / Fedu : niveau d’éducation des parents
famrel : relations familiales
health : état de santé
absences_mat : absences
G3_mat / G3_por : notes finales (maths / portugais)
failures_mat / failures_por : échecs scolaires
Cluster 0 : « Sociables mais performants »

Caractéristiques :

Consommation d’alcool assez élevée :
Dalc = 2.38
Walc = 3.73
Sortent beaucoup :
goout = 4.01
Beaucoup d’absences :
9.10
Notes correctes :
Maths = 10.16
Portugais = 11.75
Très peu d’échecs :
failures ≈ 0

Interprétation :

Ce groupe semble représenter des étudiants très sociaux, avec beaucoup d’activités et de sorties, mais qui arrivent malgré tout à maintenir des résultats acceptables.

Cluster 1 : « Profil intermédiaire / équilibré »

Caractéristiques :

Faible alcool :
Dalc = 1.92
Walc = 3.03
Santé élevée :
4.11
Peu d’absences :
4.72
Résultats moyens :
Maths = 6.19
Portugais = 8.61
Échecs plus fréquents :
1.92 en maths

Interprétation :

Étudiants plutôt équilibrés dans leurs habitudes, mais avec des performances académiques modérées.

Cluster 2 : « Travailleurs avec bons résultats en langues »

Caractéristiques :

Temps d’étude élevé :
2.37 (plus élevé)
Très faible alcool :
Dalc = 1.13
Walc = 1.76
Peu de sorties :
2.78
Absences modérées :
4.65
Portugais élevé :
12.60
Maths correcte :
9.59

Interprétation :

Ce profil ressemble à des étudiants plus disciplinés, moins tournés vers les sorties et l’alcool, avec de bonnes performances scolaires.

Cluster 3 : « Excellents étudiants »

Caractéristiques :

Très bonnes notes :
Maths = 12.81
Portugais = 14.24
Aucune difficulté scolaire :
failures ≈ 0
Bon environnement familial :
famrel = 4.16
Medu/Fedu relativement élevés
Très peu d’absences :
3.64
Faible alcool :
Dalc = 1.07
Walc = 1.59

Interprétation :

C’est clairement le groupe des étudiants les plus performants :

bons résultats
peu d’absences
peu d’alcool
soutien familial plus fort
Tendances globales visibles
L’alcool et les sorties semblent associés à des performances plus faibles
Cluster 0 : alcool/sorties élevés → résultats moyens
Cluster 3 : alcool faible → meilleures notes
Les absences semblent fortement liées aux résultats
Cluster 0 : absences = 9.10
Cluster 3 : absences = 3.64
L’éducation parentale pourrait jouer un rôle
Cluster 3 a des niveaux parentaux plus élevés (Medu/Fedu).
Les échecs passés sont fortement liés aux notes finales
Cluster 3 : presque aucun échec → meilleures notes
Cluster 1 : davantage d’échecs → moins bonnes notes

Pour un rapport ou mémoire, vous pourriez écrire quelque chose comme :

L’analyse par clustering met en évidence quatre profils distincts d’étudiants. Les clusters se différencient principalement selon les habitudes sociales (alcool, sorties), l’assiduité (absences) et les performances scolaires. Le cluster 3 regroupe les étudiants les plus performants, caractérisés par peu d’absences, une faible consommation d’alcool et un environnement familial plus favorable, tandis que le cluster 0 correspond à des étudiants plus sociables mais présentant davantage d’absences.